In [14]:
from cell_tracking.data import load_image_array, load_tracks_array
from cell_tracking.utils import view_with_tracks
import napari


arr = load_image_array(r"C:\Users\larss\Downloads\biohub-cell-tracking-during-development\train\6bba_fe670320.zarr")
tracks = load_tracks_array(r"C:\Users\larss\Downloads\biohub-cell-tracking-during-development\train\6bba_fe670320.geff")



print(arr)


viewer = view_with_tracks(arr, tracks)
napari.run()


<Array file://C:/Users/larss/Downloads/biohub-cell-tracking-during-development/train/6bba_fe670320.zarr/0 shape=(100, 64, 256, 256) dtype=uint16>


In [ ]:
from cell_tracking.data import CellDataset
from cell_tracking.data import load_image_array, load_tracks_array
from cell_tracking.utils import view_with_tracks
import numpy as np



#Test if the CellDataset class works

path = r"C:\Users\larss\Downloads\biohub-cell-tracking-during-development\train"

dataset = CellDataset(path)

img, centers, metadata = dataset[40]

for i, sample in enumerate(dataset.samples):
    print(i, sample)


print(centers)

In [6]:
import torch

from cell_tracking.models import Simple3DCellDetector
from cell_tracking.data import CellDataset
from cell_tracking.data import construct_gaussian_heatmap


path = r"C:\Users\larss\Downloads\biohub-cell-tracking-during-development\train"

dataset = CellDataset(path)

img, centers, metadata = dataset[0]

model = Simple3DCellDetector()

# NumPy -> PyTorch tensor
img = torch.from_numpy(img).float()

# Basic normalization
img = (img - img.mean()) / (img.std() + 1e-8)

# [Z, Y, X] -> [1, 1, Z, Y, X]
img = img.unsqueeze(0).unsqueeze(0)

print("Input shape:", img.shape)

prediction = model(img)

print("Prediction shape:", prediction.shape)

target = construct_gaussian_heatmap(
    shape=(64, 256, 256),
    centers=centers
)

target = torch.from_numpy(target).float()
target = target.unsqueeze(0).unsqueeze(0)

criterion = torch.nn.MSELoss()

loss = criterion(prediction, target)

print("Loss:", loss.item())

Input shape: torch.Size([1, 1, 64, 256, 256])
Prediction shape: torch.Size([1, 1, 64, 256, 256])
Loss: 0.28656700253486633


In [7]:
import torch
from torch.utils.data import Subset

from cell_tracking.models import Simple3DCellDetector
from cell_tracking.data import CellDataset, construct_gaussian_heatmap


path = r"C:\Users\larss\Downloads\biohub-cell-tracking-during-development\train"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

dataset = CellDataset(path)

# Small subset for testing
subset_size = min(30, len(dataset))
train_subset = Subset(dataset, range(subset_size))

model = Simple3DCellDetector().to(device)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

num_epochs = 3

Using device: cuda


In [8]:
for epoch in range(num_epochs):

    model.train()

    total_loss = 0.0

    for i in range(len(train_subset)):

        img, centers, metadata = train_subset[i]

        # -------------------------
        # Prepare image
        # -------------------------

        img = torch.from_numpy(img).float()

        img = (
            img - img.mean()
        ) / (
            img.std() + 1e-8
        )

        # [Z, Y, X]
        # ->
        # [1, 1, Z, Y, X]
        img = img.unsqueeze(0).unsqueeze(0)

        img = img.to(device)

        # -------------------------
        # Create target heatmap
        # -------------------------

        target = construct_gaussian_heatmap(
            shape=img.shape[-3:],
            centers=centers,
            sigma=2.0
        )

        target = torch.from_numpy(
            target
        ).float()

        target = target.unsqueeze(0).unsqueeze(0)

        target = target.to(device)

        # -------------------------
        # Forward pass
        # -------------------------

        prediction = model(img)

        loss = criterion(
            prediction,
            target
        )

        # -------------------------
        # Backpropagation
        # -------------------------

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        if i % 5 == 0:
            print(
                f"Epoch {epoch + 1}/{num_epochs} | "
                f"Sample {i}/{len(train_subset)} | "
                f"Loss: {loss.item():.6f}"
            )

    average_loss = (
        total_loss / len(train_subset)
    )

    print(
        f"\nEpoch {epoch + 1} finished | "
        f"Average loss: {average_loss:.6f}\n"
    )

Epoch 1/3 | Sample 0/30 | Loss: 0.330608
Epoch 1/3 | Sample 5/30 | Loss: 0.255336
Epoch 1/3 | Sample 10/30 | Loss: 0.131267
Epoch 1/3 | Sample 15/30 | Loss: 0.033978
Epoch 1/3 | Sample 20/30 | Loss: 0.007560
Epoch 1/3 | Sample 25/30 | Loss: 0.001990

Epoch 1 finished | Average loss: 0.104220

Epoch 2/3 | Sample 0/30 | Loss: 0.000815
Epoch 2/3 | Sample 5/30 | Loss: 0.000379
Epoch 2/3 | Sample 10/30 | Loss: 0.000244
Epoch 2/3 | Sample 15/30 | Loss: 0.000179
Epoch 2/3 | Sample 20/30 | Loss: 0.000145
Epoch 2/3 | Sample 25/30 | Loss: 0.000116

Epoch 2 finished | Average loss: 0.000259

Epoch 3/3 | Sample 0/30 | Loss: 0.000121
Epoch 3/3 | Sample 5/30 | Loss: 0.000109
Epoch 3/3 | Sample 10/30 | Loss: 0.000106
Epoch 3/3 | Sample 15/30 | Loss: 0.000102
Epoch 3/3 | Sample 20/30 | Loss: 0.000098
Epoch 3/3 | Sample 25/30 | Loss: 0.000086

Epoch 3 finished | Average loss: 0.000102

